## Introduction to Data Cleaning : Part 1

### What is Data Cleaning

- Data Cleaning is the process of fixing or removing incorrect ,corrupted ,incorrectly formatted ,duplicated or incomplete data within a dataset

### Why do we need to clean data?

Before building models, dashboards, or analytics, raw data is usually:
*    Incomplete
*    Inconsistent
*    Incorrect
*    Duplicated
*    Messy

If we don’t clean it:
* Analysis becomes misleading
* Models learn wrong patterns
* Business decisions become risky

<img src="img.png" width="500">

### Example

In [2]:
import pandas as pd

In [3]:
clean=pd.read_csv("clean_dataset.csv")
clean.mean()

Hours_Studied     5.84
Test_Score       73.76
dtype: float64

In [4]:
unclean=pd.read_csv("unclean_dataset.csv")
unclean.mean()

TypeError: can only concatenate str (not "int") to str

In [5]:
unclean.dtypes

Hours_Studied     object
Test_Score       float64
dtype: object

In [6]:
clean.dtypes

Hours_Studied    int64
Test_Score       int64
dtype: object

### Steps to follow when cleaning your data

Before you cleaning ,you inspect first and to do so you do the following:
- Check what are the columns
- Check the datatypes
- What does the data represent

The aim is to understand what you need to fix before fixing it


### Step 1:Changing Datatypes

* Numeric operations won’t work on strings
* Dates as strings cannot be used for time analysis
* Categorical data stored as numbers might be misinterpreted

1. Numeric Conversion
      ```python
 data["Hours_Studied"] = pd.to_numeric(data["Hours_Studied"], errors="coerce")
           data["Test_Score"] = pd.to_numeric(data["Test_Score"], errors="coerce")
      ```
  - errors="coerce" converts invalid values (like "six") to NaN

2. Categorical Conversion
   ```python
  data["Student_ID"] = data["Student_ID"].astype("category")
      ```
3. Date Conversion
   ```python
   data["Exam_Date"] = pd.to_datetime(data["Exam_Date"], errors="coerce")
```

### Step 2: Check for missing values
```python
data.isnull().sum()
```
Then you will need to decide what time of missingness is this
* Missing Completely at Random → Simple fix
* Missing At Random → Smart imputation
* Missing Not at Random → Be cautious / investigate

<img src="types_of _missing.png" width="400">

#### Example

In [23]:
import numpy as np

data_num = pd.DataFrame({
    "Student_ID": [1,2,3,4,5,6,7,8,9,10],
    "Hours_Studied": [5,8,4,6,np.nan,7,2,9,np.nan,3],
    "Test_Score": [70,85,np.nan,75,30,90,np.nan,95,15,40]
})

In [24]:
data_num.isnull().sum()

Student_ID       0
Hours_Studied    2
Test_Score       2
dtype: int64

In [25]:
data_num[data_num.isnull().any(axis=1)]

,Student_ID,Hours_Studied,Test_Score
2,3,4.0,NaN
4,5,NaN,30.0
6,7,2.0,NaN
8,9,NaN,15.0


### How to handle Missing Completely at Random Data (Numeric)

1. Remove Missing Rows
  - if missing entries make less than 5% of your data then itis recommended that yuo remove those rows
    - ```python
      data_cleaned = data.dropna()
     ```

2. Remove Only Specific Column Missing Values
   - If the column has 80% of the data missing and has no relevance to your problem statement then remove it
      - ```python
        data = data.dropna(subset=["Student_ID"])
        ```
3. Mean Imputation (For Numeric Data)
   - Replace the missing values with the column mean if your data is symmetric
      - ```python
           data["Test_Score"].fillna(data["Test_Score"].mean(), inplace=True)
      ```
4. Median Imputation (Better for Outliers)
    - If data is skewed you mean( we will explain more on this later when we get to statistical analysis
      - ```python
        data["Test_Score"].fillna(data["Test_Score"].median(), inplace=True)
     ```

### Handling MCAR for Categorical Data

In [18]:
data = pd.DataFrame({
    "Student_ID": [1,2,3,4,5,6,7,8,9,10],
    "Gender": ["M","F",np.nan,"F","M","F","M","F",np.nan,"M"],
    "Test_Score": [70,85,88,75,60,90,65,95,50,40]
})


In [19]:
data[data.isnull().any(axis=1)]

,Student_ID,Gender,Test_Score
2,3,NaN,88
8,9,NaN,50


In [21]:
print("\nMost frequent category:", data["Gender"].mode()[0])


Most frequent category: F


In [22]:
data['Gender'].value_counts(normalize=True)

Gender
M    0.5
F    0.5
Name: proportion, dtype: float64

1. Remove Missing Rows
  - if missing entries make less than 5% of your data then itis recommended that yuo remove those rows
    - ```python
      data_cleaned = data.dropna()
     ```

2. Impute With Most Frequent Category (Mode)
  - Replaces missing values with the most common category
     - ```python
       mode_gender = data["Gender"].mode()[0]
       data["Gender"].fillna(mode_gender, inplace=True)
      ```

3. Impute With a New Category
  - Sometimes it’s better to create a “Missing” category
    - ```python
       data["Gender"].fillna("Missing", inplace=True)
      ```


### Handling Missing At Random (MAR) Data

Since the missingness depends on another variable, simple mean/median imputation may bias your dataset

1. Group-Based Imputation
  - Fill missing values using mean/median of the group based on a related variable.
    - ```python
      data["Hours_Studied"] = data.groupby(pd.cut(data["Test_Score"], bins=[0,60,80,100]))["Hours_Studied"]\
              .transform(lambda x: x.fillna(x.mean()))
      ```


In [30]:
data = pd.DataFrame({
    "Student_ID": range(1, 11),
    "Preferred_Study_Method": ["Group","Alone",np.nan,"Alone","Online","Group",np.nan,"Online","Group","Alone"],
    "Test_Score": [70,85,60,75,50,90,65,95,55,40]
})

data

,Student_ID,Preferred_Study_Method,Test_Score
0,1,Group,70
1,2,Alone,85
2,3,NaN,60
3,4,Alone,75
4,5,Online,50
5,6,Group,90
6,7,NaN,65
7,8,Online,95
8,9,Group,55
9,10,Alone,40


### Handling Missing At Random (MAR) for Categorical Data
1. Group-Based Imputation (Mode per Group)
  ```python
bins = [0, 60, 80, 100]
data["Score_Group"] = pd.cut(data["Test_Score"], bins=bins)

# Fill missing with mode of the group
data["Preferred_Study_Method"] = data.groupby("Score_Group")["Preferred_Study_Method"]\
                                     .transform(lambda x: x.fillna(x.mode()[0]))

```

### Step 3 : Handling Duplicates

* Duplicate rows can distort averages, counts, correlations
* Can bias machine learning models
* Often happens when combining multiple datasets

1. Remove Duplicate Rows
  ```python
data_cleaned = data.drop_duplicates()
```

In [32]:
data = pd.DataFrame({
    "Student_ID": [1,2,3,4,5,5,6,7,7,8],
    "Hours_Studied": [5,8,4,6,7,7,9,3,3,10],
    "Test_Score": [70,85,60,75,80,80,90,50,50,95]
})

In [34]:
data.duplicated().sum()

np.int64(2)